In [ ]:
# =============================================================================
# MAIN ORCHESTRATOR: Sync OHLCV -> Features -> Predictions -> View
# =============================================================================

import time
import sqlite3
import pandas as pd
import sys
import os

sys.path.insert(0, "..")
import utils
from database_codes.sync_ohlcv import sync_ohlcv
from database_codes.features import sync_features
from database_codes.predictions import sync_predictions
from database_codes.pred_view import display_predictions

# =============================================================================
# CONFIGURATION
# =============================================================================
POLL_SECONDS       = 60
SEPARATOR          = "=" * 80
INIT_START_DATE    = "2017-01-01 00:00:00"
LOOKBACK_MINUTES   = 240

# =============================================================================
# get_last_timestamp(db_path: str, table_name: str) -> str
# =============================================================================
# Purpose:
#  - Query table MAX(open_time)
#  - If table doesn't exist or is empty, return None
#  - Return timestamp string or None
# Parameters:
#  - db_path: database path
#  - table_name: name of table to query
# =============================================================================
def get_last_timestamp(db_path: str, table_name: str) -> str:
	try:
		with sqlite3.connect(db_path) as conn:
			result = pd.read_sql_query(
				f"SELECT MAX(open_time) as max_time FROM {table_name}",
				conn
			)
		max_time = result["max_time"].iloc[0]
		return max_time
	except:
		return None


# =============================================================================
# MAIN: Orchestration loop
# =============================================================================
# Purpose:
#  - Poll database every POLL_SECONDS
#  - Sync OHLCV (init with INIT_START_DATE if empty)
#  - Sync Features (only if OHLCV has new data)
#  - Sync Predictions (only if Features has new data)
#  - Display predictions chart (skip if no data)
# =============================================================================
def main_loop():
	config = utils._load_config()
	db_path = config["database"]["db_path"]

	table_ohlcv = config["database"]["tables"]["ohlcv"]
	table_feat  = config["database"]["tables"]["features"]
	table_pred  = config["database"]["tables"]["predictions"]

	cycle = 1

	while True:
		print(f"\n{SEPARATOR}")
		print(f"🔄 Cycle #{cycle} at {utils.now_utc_str()}")
		print(SEPARATOR)

		# -----
		# OHLCV
		# -----
		print("📥 OHLCV SECTION")
		max_ohlcv = get_last_timestamp(db_path, table_ohlcv)

		if max_ohlcv:
			print(f"   Last: {max_ohlcv}")
			start_ms = int(pd.to_datetime(max_ohlcv).timestamp() * 1000) + 60000
		else:
			print(f"   Table empty. Initializing from {INIT_START_DATE}...")
			start_ms = int(pd.to_datetime(INIT_START_DATE).timestamp() * 1000)

		sync_ohlcv(start_ms)

		# --------
		# FEATURES
		# --------
		print("\n📊 FEATURES SECTION")
		max_feat      = get_last_timestamp(db_path, table_feat)
		max_ohlcv_now = get_last_timestamp(db_path, table_ohlcv)

		if max_ohlcv_now:
			if max_feat is None:
				# Features empty, start from INIT_START_DATE
				start_feat = INIT_START_DATE
				print(f"   Table empty. Initializing from {start_feat}...")
				sync_features(start_feat)
			elif max_feat < max_ohlcv_now:
				# Features outdated, sync new data
				start_feat = (pd.to_datetime(max_feat) + pd.Timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S")
				print(f"   Last: {max_feat}")
				sync_features(start_feat)
			else:
				print(f"   Last: {max_feat} (up to date)")
		else:
			print(f"   No OHLCV data yet. Skipping.")

		# -----------
		# PREDICTIONS
		# -----------
		print("\n🤖 PREDICTIONS SECTION")
		max_pred     = get_last_timestamp(db_path, table_pred)
		max_feat_now = get_last_timestamp(db_path, table_feat)

		if max_feat_now:
			if max_pred is None:
				# Predictions empty, start from INIT_START_DATE
				start_pred = INIT_START_DATE
				print(f"   Table empty. Initializing from {start_pred}...")
				sync_predictions(start_pred)
			elif max_pred < max_feat_now:
				# Predictions outdated, sync new data
				start_pred = (pd.to_datetime(max_pred) + pd.Timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S")
				print(f"   Last: {max_pred}")
				sync_predictions(start_pred)
			else:
				print(f"   Last: {max_pred} (up to date)")
		else:
			print(f"   No Features data yet. Skipping.")

		# ----
		# VIEW
		# ----
		print("\n📈 VIEW SECTION")
		display_predictions()

		# -----
		# WAIT
		# -----
		print(f"\n{SEPARATOR}")
		print(f"✅ Cycle #{cycle} complete. Sleeping {POLL_SECONDS}s...")
		print(SEPARATOR)

		cycle += 1
		time.sleep(POLL_SECONDS)


# =============================================================================
# ENTRYPOINT
# =============================================================================
if __name__ == "__main__":
	main_loop()